# Knee MRI v2 -- Own DICOM Preprocessing, Shared Backbone, Per-Plane Pooling

Same model design as v1 (shared EfficientNet-B0, per-plane attention pooling, one joint
12-label head -- see [docs/architecture_research.md](../../docs/architecture_research.md)
and [docs/model_architecture_plan.md](../../docs/model_architecture_plan.md)). What's new is
**Section A**: a real, research-informed DICOM preprocessing pipeline that replaces the two
things that broke v1's submission:

1. `NPZ_DIR` was a static third-party dataset ([barun2104/rsna-knee-mri-processed-3d-volumes](https://www.kaggle.com/datasets/barun2104/rsna-knee-mri-processed-3d-volumes))
   that can only contain series that were public before it was made -- it can never contain
   the ~1300 hidden test studies, so every test prediction came out identical (AUC exactly 0.5).
2. The ad-hoc DICOM fallback bolted onto v1's feature-extraction step used only
   `InstanceNumber`/filename ordering and per-slice normalization -- workable, but not the
   real standard.

v2 preprocesses the **official raw DICOM** (`train_series/` + `test_series/`,
`<StudyUID>/<SeriesUID>/<SOPUID>.dcm`) once, up front, for train *and* test alike, into a
single self-owned cache -- so "the model gets the proper data" is true by construction, not
by hoping a third-party dataset happens to cover whatever gets asked of it.

## Data note (verified against your downloaded `Raw_dicom`, not guessed)

A study folder having 5 series subfolders does **not** mean there are 5 planes -- there are
still only 3 (Sagittal / Coronal / Axial). Checked directly against
`1.2.826.0.1.3680043.8.498.10004873229099053869093324292195817260`, which has exactly 5
series: **2 Sagittal, 2 Coronal, 1 Axial** -- one plane got a plain acquisition and a
fluid-sensitive/fat-sat one, matching the `Fluid_Sensitive`/`Fat_Suppression` columns in
`train_series.csv`. Series-per-plane varies per study (1-7 in the full dataset); 5 total
series per study is just the most common count, not a fixed layout.

## Preprocessing design (researched, not guessed) and why

- **Slice ordering**: project `ImagePositionPatient` onto the normal of
  `ImageOrientationPatient` (`normal = cross(row_dir, col_dir)`, sort by `dot(normal, position)`)
  -- the spatially-correct order used by real DICOM viewers and by
  [JunhaoLiXD/RSNA_Knee_Abnormality_Detection](https://github.com/JunhaoLiXD/RSNA_Knee_Abnormality_Detection),
  a real, working baseline for this exact competition. Falls back to `InstanceNumber`, then
  filename, if geometry tags are missing.
- **Rescale + photometric handling**: applies `RescaleSlope`/`RescaleIntercept`, inverts
  `MONOCHROME1` series (bright/dark reversed vs. the usual `MONOCHROME2`), and takes the
  middle frame for any multi-frame `.dcm` -- all things the naive v1 fallback did not handle
  and could have silently gotten wrong.
- **Normalization**: MRI has no fixed windowing convention the way CT does (no Hounsfield
  units), so per-series 1st/99th-percentile clipping -> scale to `[0, 255]` uint8 is used --
  the same approach as JunhaoLiXD's real baseline. The classic
  [MRNet](https://stanfordmlgroup.github.io/projects/mrnet/) baseline (Stanford) instead used
  full histogram-matching normalization and 256x256 PNGs; percentile clipping is simpler and
  is what v1 already validated (0.75 honest AUC on real data), so it's kept rather than adding
  the extra complexity of histogram standardization for a v2 whose main job is fixing the
  test-time data gap, not re-deriving normalization from scratch.
- **Library**: `pydicom` only (already installed in both `tf` and `torch_gpu`). SimpleITK and
  MONAI came up in research as the more "batteries-included" options for DICOM series
  handling, but both are heavier dependencies not confirmed pre-installed on Kaggle -- and we
  already lost a submission once to a dependency that needed the internet at commit time
  (EfficientNet-B0's pretrained weights, fixed in v1). `pydicom` + a ~15-line ordering formula
  covers everything actually needed here without repeating that mistake.
- **Storage format**: **uncompressed** `.npy` per series, not `.npz`/`savez_compressed`. This
  is a direct lesson from v1's real profiling data -- zlib-compressed `.npz` was the dominant
  bottleneck (441-600s/epoch), and the whole preprocessed training set is only ~18-20GB
  uncompressed at `(N_SLICES, 224, 224)` uint8, comfortably inside Kaggle's working-directory
  quota. "Efficient" here means fast to decode, not maximally small.
- **Metadata is kept, not discarded**: converting DICOM to plain images throws away
  everything except pixels unless you capture it separately. `preprocessing_metadata.csv`
  keeps `PixelSpacing`, `SliceThickness`, `Manufacturer`, `MagneticFieldStrength`,
  `PhotometricInterpretation`, and decode-success counts per series. Scanner/manufacturer
  fields are captured for auditing and possible scanner-grouped CV later -- deliberately
  **not** fed into the model as input features, since a model keying on scanner identity
  instead of anatomy is a classic shortcut-learning trap, not a real diagnostic signal.

## Two-stage cache, kept separate on purpose

```
Section A (NEW): raw DICOM -> normalized (N_SLICES, 224, 224) uint8 .npy per series
                 + preprocessing_metadata.csv          [model-agnostic, run once]
Section B (from v1): those arrays -> frozen EfficientNet-B0 features, cached per study
                 [model-specific -- swap backbones later without re-touching DICOM at all]
```

Everything from Section C onward (loss, training loop, per-fold CV, OOF summary, submission)
is unchanged from v1 -- only the data source changed, so the 0.75 honest-AUC result stays a
fair comparison point.

In [1]:
import os
import time
import random
import threading
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.metrics import roc_auc_score
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
import pydicom  

TARGET_COLS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture',
]
PLANES = ['Sagittal', 'Coronal', 'Axial']

In [2]:
# -----------------------------------------------------------------------
# Config -- IS_KAGGLE auto-detects the environment; only DATA_DIR needs a manual edit on
# Kaggle (point it at your dataset mount). Both TRAIN/TEST_DICOM_DIR use the identical
# official <StudyUID>/<SeriesUID>/<SOPUID>.dcm layout locally and on Kaggle.
# -----------------------------------------------------------------------
IS_KAGGLE = os.path.isdir('/kaggle')

DATA_DIR = 'D:/Coding/kaggle/RSNA/data'   # <-- on Kaggle, change this to your dataset mount path
MAIN_DIR = f'{DATA_DIR}/main_data'

CONSENSUS_LABELS_CSV = f'{MAIN_DIR}/final/train_folds_consensus.csv'
SERIES_META_CSV = f'{MAIN_DIR}/train_series.csv'
GOLD_LABELS_CSV = f'{MAIN_DIR}/to_use/available_labels.csv'
TEST_CSV = f'{MAIN_DIR}/test.csv'
TEST_SERIES_CSV = f'{MAIN_DIR}/test_series.csv'
SAMPLE_SUB_CSV = f'{MAIN_DIR}/sample_submission.csv'

if IS_KAGGLE:
    KAGGLE_COMP_DIR = '/kaggle/input/competitions/rsna-knee-abnormality-detection'
    TRAIN_DICOM_DIR = f'{KAGGLE_COMP_DIR}/train_series'
    TEST_DICOM_DIR = f'{KAGGLE_COMP_DIR}/test_series'
else:
    TRAIN_DICOM_DIR = f'{DATA_DIR}/Raw_dicom/train_series'
    TEST_DICOM_DIR = f'{DATA_DIR}/Raw_dicom/test_series'

OUTPUT_DIR = '/kaggle/working' if IS_KAGGLE else 'D:/Coding/kaggle/RSNA/output/v2'
PREPROCESSED_DIR = f'{OUTPUT_DIR}/preprocessed'     # Section A output: one (T,224,224) uint8 .npy per series
FEATURE_CACHE_DIR = f'{OUTPUT_DIR}/feature_cache'   # Section B output: one backbone-feature .npz per study
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PREPROCESSED_DIR, exist_ok=True)
os.makedirs(FEATURE_CACHE_DIR, exist_ok=True)

# See v1's Section 2 note -- required so the internet-off Kaggle commit run doesn't crash
# trying to download pretrained weights. Leave None until you've set this up once.
EFFICIENTNET_WEIGHTS_PATH = None   # e.g. '/kaggle/input/efficientnet-b0-weights/efficientnet_b0_weights.pth'

# Data / model shape
N_SLICES = 12
IMG_SIZE = 224
FREEZE_BACKBONE = True

# Preprocessing + precompute (one-time, CPU-heavy)
PREPROCESS_BATCH_SIZE = 16
NUM_LOAD_WORKERS = max(1, (os.cpu_count() or 2) - 1)   # leave 1 core free for the OS/notebook

# Training (fast, on cached feature vectors)
TRAIN_BATCH_SIZE = 32
TRAIN_NUM_WORKERS = 0
EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 1e-4
GOLD_WEIGHT = 8.0
USE_AMP = torch.cuda.is_available()

# [0] = fast sanity run. Use list(range(5)) for the real, full CV run.
FOLDS_TO_RUN = [0]

SEED = 42


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'} | OUTPUT_DIR: {OUTPUT_DIR}")
print(f'Device: {device} | loader threads: {NUM_LOAD_WORKERS} (of {os.cpu_count()} logical cores)')

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

Environment: Local | OUTPUT_DIR: D:/Coding/kaggle/RSNA/output/v2
Device: cuda | loader threads: 11 (of 12 logical cores)


## 1. Load series metadata (train + test)

In [3]:
train_series_df = pd.read_csv(SERIES_META_CSV)
test_series_df = pd.read_csv(TEST_SERIES_CSV)
consensus_df = pd.read_csv(CONSENSUS_LABELS_CSV)
gold_df = pd.read_csv(GOLD_LABELS_CSV)
gold_ids = set(gold_df['StudyInstanceUID'])

print(f'Train studies: {consensus_df["StudyInstanceUID"].nunique()} | train series: {len(train_series_df)}')
print(f'Test studies: {test_series_df["StudyInstanceUID"].nunique()} | test series: {len(test_series_df)} '
      f'(the public stub -- real run substitutes ~1300 studies here)')
print(f'Gold rows: {len(gold_ids)}')


def build_priority_series(series_meta):
    """One series per (study, plane): prefer Fluid_Sensitive==1 (fat-sat), else any."""
    priority = {}
    series_meta = series_meta.sort_values('SeriesInstanceUID')
    for (uid, plane), grp in series_meta.groupby(['StudyInstanceUID', 'Anatomical_Plane']):
        fluid = grp[grp['Fluid_Sensitive'] == 1]
        chosen = fluid.iloc[0] if len(fluid) else grp.iloc[0]
        priority[(uid, plane)] = chosen['SeriesInstanceUID']
    return priority


train_priority_series = build_priority_series(train_series_df)
test_priority_series = build_priority_series(test_series_df)
print(f'Train priority series: {len(train_priority_series)} | test priority series: {len(test_priority_series)}')

Train studies: 4407 | train series: 24371
Test studies: 3 | test series: 15 (the public stub -- real run substitutes ~1300 studies here)
Gold rows: 58
Train priority series: 13221 | test priority series: 9


## Section A: DICOM preprocessing

Reads every series once (train + test), orders slices correctly, applies rescale/photometric
correction, normalizes, resizes, and writes one `.npy` per series to `PREPROCESSED_DIR` --
plus a metadata CSV. Parallelized across `NUM_LOAD_WORKERS` threads (not processes -- same
reasoning as v1: Windows/Jupyter can't pickle notebook-defined functions into spawned worker
processes; threads sidestep that, and file I/O + DICOM decode both release the GIL).

In [4]:
def _order_series_files(series_dir):
    """Return [(filename, pydicom.Dataset), ...] in spatially-correct order."""
    fnames = [f for f in os.listdir(series_dir) if f.lower().endswith('.dcm')]
    entries = []
    for fname in fnames:
        try:
            ds = pydicom.dcmread(os.path.join(series_dir, fname))
        except Exception:
            continue
        entries.append([None, fname, ds])
    if not entries:
        return []

    # 1) Spatially correct: project ImagePositionPatient onto the plane normal derived from
    #    ImageOrientationPatient -- the same order a real DICOM viewer reconstructs.
    have_geometry = True
    for e in entries:
        try:
            iop = np.array(e[2].ImageOrientationPatient, dtype=np.float64)
            ipp = np.array(e[2].ImagePositionPatient, dtype=np.float64)
            normal = np.cross(iop[0:3], iop[3:6])
            e[0] = float(np.dot(normal, ipp))
        except Exception:
            have_geometry = False
            break
    if have_geometry and len({round(e[0], 3) for e in entries}) > 1:
        entries.sort(key=lambda e: e[0])
        return [(e[1], e[2]) for e in entries]

    # 2) Fallback: InstanceNumber, if present and distinct for every slice.
    try:
        nums = [int(getattr(e[2], 'InstanceNumber', -1)) for e in entries]
        if len(set(nums)) == len(entries) and all(n >= 0 for n in nums):
            order = sorted(range(len(entries)), key=lambda i: nums[i])
            return [(entries[i][1], entries[i][2]) for i in order]
    except Exception:
        pass

    # 3) Last resort: filename order (imperfect, but the model attention-pools over slices
    #    rather than treating them as a strict sequence, so this is a soft degradation).
    entries.sort(key=lambda e: e[1])
    return [(e[1], e[2]) for e in entries]


def _decode_slice(ds):
    """One DICOM Dataset -> float32 2D array, rescaled and photometric-corrected."""
    arr = ds.pixel_array.astype(np.float32)
    if arr.ndim == 3:                 # multi-frame file -- take the middle frame
        arr = arr[arr.shape[0] // 2]
    slope = float(getattr(ds, 'RescaleSlope', 1))
    intercept = float(getattr(ds, 'RescaleIntercept', 0))
    arr = arr * slope + intercept
    if str(getattr(ds, 'PhotometricInterpretation', '')) == 'MONOCHROME1':
        arr = arr.max() - arr          # MONOCHROME1 is inverted vs. the usual MONOCHROME2
    return arr


def _tag(ds, name, cast=None, default=None):
    if not hasattr(ds, name):
        return default
    try:
        val = getattr(ds, name)
        return cast(val) if cast else val
    except Exception:
        return default


def preprocess_series(series_dir, img_size):
    """One series directory -> ((T, img_size, img_size) uint8 array, metadata dict) or (None, {})."""
    ordered = _order_series_files(series_dir)
    if not ordered:
        return None, {}

    slices = []
    for _, ds in ordered:
        try:
            slices.append(_decode_slice(ds))
        except Exception:
            continue                  # unreadable slice (e.g. missing JPEG2000 plugin) -- skip, don't fail the series
    if not slices:
        return None, {}

    stack = np.stack(slices, axis=0)
    lo, hi = np.percentile(stack, [1.0, 99.0])   # per-series normalization -- MRI has no fixed windowing
    hi = max(hi, lo + 1.0)
    stack = np.clip((stack - lo) / (hi - lo), 0, 1) * 255.0
    stack = stack.astype(np.uint8)

    resized = np.stack([
        np.array(Image.fromarray(s).resize((img_size, img_size), Image.BILINEAR), dtype=np.uint8)
        for s in stack
    ])

    first_ds = ordered[0][1]
    spacing = _tag(first_ds, 'PixelSpacing')
    meta = {
        'n_dcm_files': len(ordered),
        'n_slices_decoded': len(slices),
        'native_rows': _tag(first_ds, 'Rows', int),
        'native_cols': _tag(first_ds, 'Columns', int),
        'pixel_spacing_row': float(spacing[0]) if spacing is not None else None,
        'pixel_spacing_col': float(spacing[1]) if spacing is not None else None,
        'slice_thickness': _tag(first_ds, 'SliceThickness', float),
        'manufacturer': _tag(first_ds, 'Manufacturer', str, ''),
        'manufacturer_model': _tag(first_ds, 'ManufacturerModelName', str, ''),
        'magnetic_field_strength': _tag(first_ds, 'MagneticFieldStrength', float),
        'photometric_interpretation': _tag(first_ds, 'PhotometricInterpretation', str, ''),
    }
    return resized, meta

In [5]:
def preprocess_all_series(series_df, dicom_root, out_dir, img_size, num_workers, skip_existing=True):
    rows = list(series_df.itertuples())
    todo = [r for r in rows
            if not (skip_existing and os.path.exists(os.path.join(out_dir, f'{r.SeriesInstanceUID}.npy')))]
    print(f'{len(todo)} / {len(rows)} series need preprocessing ({num_workers} threads) -> {dicom_root}')
    if not todo:
        return []

    def _work(row):
        series_dir = os.path.join(dicom_root, row.StudyInstanceUID, row.SeriesInstanceUID)
        if not os.path.isdir(series_dir):
            return row.SeriesInstanceUID, None, {}
        arr, meta = preprocess_series(series_dir, img_size)
        return row.SeriesInstanceUID, arr, meta

    meta_rows = []
    t0 = time.time()
    done = 0
    failed = 0
    with ThreadPoolExecutor(max_workers=num_workers) as pool:
        for series_uid, arr, meta in pool.map(_work, todo):
            if arr is not None:
                np.save(os.path.join(out_dir, f'{series_uid}.npy'), arr)
            else:
                failed += 1
            meta['SeriesInstanceUID'] = series_uid
            meta['ok'] = arr is not None
            meta_rows.append(meta)
            done += 1
            if done % 200 == 0 or done == len(todo):
                elapsed = time.time() - t0
                rate = done / elapsed if elapsed > 0 else 0.0
                print(f'  {done}/{len(todo)} | {elapsed:.0f}s | {rate:.2f} series/s | {failed} failed so far')

    print(f'Done: {done} series in {time.time() - t0:.0f}s, {failed} failed -> {out_dir}')
    if failed > 0:
        print(f'WARNING: {failed} series produced no usable slices -- studies relying on them '
              f'will fall back to an all-zero plane in Section B. Check preprocessing_metadata.csv.')
    return meta_rows

In [6]:
# Run once (Section A): preprocesses every train + test series into PREPROCESSED_DIR.
# Safe to re-run -- skip_existing=True skips series already cached.
train_meta = preprocess_all_series(train_series_df, TRAIN_DICOM_DIR, PREPROCESSED_DIR,
                                    IMG_SIZE, NUM_LOAD_WORKERS)
test_meta = preprocess_all_series(test_series_df, TEST_DICOM_DIR, PREPROCESSED_DIR,
                                   IMG_SIZE, NUM_LOAD_WORKERS, skip_existing=False)

meta_df = pd.DataFrame(train_meta + test_meta)
if len(meta_df):
    meta_path = os.path.join(PREPROCESSED_DIR, 'preprocessing_metadata.csv')
    meta_df.to_csv(meta_path, index=False)
    print(f'\nWrote metadata for {len(meta_df)} series -> {meta_path}')
    print(meta_df['ok'].value_counts())

0 / 24371 series need preprocessing (11 threads) -> D:/Coding/kaggle/RSNA/data/Raw_dicom/train_series
15 / 15 series need preprocessing (11 threads) -> D:/Coding/kaggle/RSNA/data/Raw_dicom/test_series
  15/15 | 8s | 1.94 series/s | 0 failed so far
Done: 15 series in 8s, 0 failed -> D:/Coding/kaggle/RSNA/output/v2/preprocessed

Wrote metadata for 15 series -> D:/Coding/kaggle/RSNA/output/v2/preprocessed\preprocessing_metadata.csv
ok
True    15
Name: count, dtype: int64


## Section B: precompute frozen-backbone features

Same idea as v1's Section 3, simplified: every series now lives in one place
(`PREPROCESSED_DIR`), decoded and normalized once in Section A, so there's no more
npz-then-DICOM branching here -- just load the `.npy`, run the frozen backbone, cache the
per-slice features. The GPU-bound backbone forward pass stays sequential on the main thread
(one GPU, nothing to parallelize there); only the file reads are threaded.

In [7]:
class SliceBackbone(nn.Module):
    def __init__(self, freeze=True):
        super().__init__()
        if EFFICIENTNET_WEIGHTS_PATH and os.path.exists(EFFICIENTNET_WEIGHTS_PATH):
            m = efficientnet_b0(weights=None)
            m.load_state_dict(torch.load(EFFICIENTNET_WEIGHTS_PATH, map_location='cpu'))
        else:
            m = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)   # needs internet
        self.features = m.features
        self.pool = m.avgpool
        self.out_dim = 1280
        if freeze:
            for p in self.features.parameters():
                p.requires_grad = False

    def forward(self, x):        # x: (N, 3, H, W)
        f = self.features(x)
        return self.pool(f).flatten(1)   # (N, out_dim)


class AttentionPool(nn.Module):
    def __init__(self, dim, hidden=128):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))

    def forward(self, x):        # x: (B, T, dim)
        w = torch.softmax(self.attn(x), dim=1)
        return (x * w).sum(dim=1)   # (B, dim)

In [8]:
_fallback_lock = threading.Lock()
_fallback_count = [0]


def _load_plane(uid, plane, priority_series):
    series_uid = priority_series.get((uid, plane))
    if series_uid is None:
        return None
    path = os.path.join(PREPROCESSED_DIR, f'{series_uid}.npy')
    if not os.path.exists(path):
        return None
    arr = np.load(path).astype(np.float32) / 255.0   # (T, H, W)
    T = arr.shape[0]
    if T > N_SLICES:
        idx = np.linspace(0, T - 1, N_SLICES).astype(int)
        arr = arr[idx]
    elif T < N_SLICES:
        pad = np.zeros((N_SLICES - T, IMG_SIZE, IMG_SIZE), dtype=np.float32)
        arr = np.concatenate([arr, pad], axis=0)
    return arr


def _load_study_planes(uid, priority_series):
    out = {}
    for plane, key in zip(PLANES, ['sagittal', 'coronal', 'axial']):
        arr = _load_plane(uid, plane, priority_series)
        if arr is None:
            with _fallback_lock:
                _fallback_count[0] += 1
            arr = np.zeros((N_SLICES, IMG_SIZE, IMG_SIZE), dtype=np.float32)
        out[key] = arr
    return uid, out


@torch.no_grad()
def precompute_features(study_ids, priority_series, backbone, cache_dir,
                         batch_size=PREPROCESS_BATCH_SIZE, num_load_workers=NUM_LOAD_WORKERS,
                         skip_existing=True):
    todo = [u for u in study_ids if not (skip_existing and os.path.exists(os.path.join(cache_dir, f'{u}.npz')))]
    print(f'{len(todo)} / {len(study_ids)} studies need feature extraction '
          f'({num_load_workers} loader threads, batch size {batch_size})')
    if not todo:
        return

    backbone.eval()
    t0 = time.time()
    done = 0
    _fallback_count[0] = 0

    with ThreadPoolExecutor(max_workers=num_load_workers) as pool:
        for start in range(0, len(todo), batch_size):
            batch_ids = todo[start: start + batch_size]
            results = list(pool.map(lambda u: _load_study_planes(u, priority_series), batch_ids))

            feats = {}
            for key in ['sagittal', 'coronal', 'axial']:
                raw = np.stack([r[1][key] for r in results])                  # (B, T, H, W)
                x = torch.from_numpy(raw).unsqueeze(2).repeat(1, 1, 3, 1, 1)   # (B, T, 3, H, W)
                x = (x - IMAGENET_MEAN) / IMAGENET_STD
                x = x.to(device)
                B, T, C, H, W = x.shape
                feats[key] = backbone(x.view(B * T, C, H, W)).view(B, T, -1).cpu().numpy()

            for b, (uid, _) in enumerate(results):
                np.savez(os.path.join(cache_dir, f'{uid}.npz'),
                         sagittal=feats['sagittal'][b], coronal=feats['coronal'][b], axial=feats['axial'][b])

            done += len(batch_ids)
            if (start // batch_size) % 20 == 0 or done == len(todo):
                elapsed = time.time() - t0
                rate = done / elapsed if elapsed > 0 else 0.0
                print(f'  {done}/{len(todo)} studies | {elapsed:.0f}s elapsed | {rate:.2f} studies/s')

    print(f'Precompute finished: {done} studies in {time.time() - t0:.0f}s -> {cache_dir}')
    if _fallback_count[0] > 0:
        print(f'WARNING: {_fallback_count[0]} / {len(todo) * 3} plane-loads fell back to an all-zero '
              f'array (missing from PREPROCESSED_DIR). Check Section A\'s failure count.')

In [9]:
class KneeFeatureDataset(Dataset):
    """Loads cached (T, 1280) per-plane backbone features for one study."""

    def __init__(self, study_ids, cache_dir, labels_df=None, gold_ids=None, gold_weight=1.0):
        self.study_ids = list(study_ids)
        self.cache_dir = cache_dir
        self.gold_ids = gold_ids or set()
        self.gold_weight = gold_weight
        self.label_lookup = None
        if labels_df is not None:
            idx = labels_df.set_index('StudyInstanceUID')[TARGET_COLS]
            self.label_lookup = {uid: idx.loc[uid].astype(float).values for uid in self.study_ids}

    def __len__(self):
        return len(self.study_ids)

    def __getitem__(self, idx):
        uid = self.study_ids[idx]
        d = np.load(os.path.join(self.cache_dir, f'{uid}.npz'))
        item = {
            'study_uid': uid,
            'sagittal': torch.from_numpy(d['sagittal']).float(),
            'coronal': torch.from_numpy(d['coronal']).float(),
            'axial': torch.from_numpy(d['axial']).float(),
        }
        if self.label_lookup is not None:
            item['labels'] = torch.tensor(self.label_lookup[uid], dtype=torch.float32)
            item['weight'] = torch.tensor(self.gold_weight if uid in self.gold_ids else 1.0, dtype=torch.float32)
        return item


class KneeFeatureModel(nn.Module):
    """Trains only the attention-pool + head on precomputed, frozen backbone features."""

    def __init__(self, feat_dim=1280, n_targets=12):
        super().__init__()
        self.slice_pool = AttentionPool(feat_dim)
        self.head = nn.Sequential(
            nn.Linear(feat_dim * 3, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, n_targets),
        )

    def forward(self, sagittal, coronal, axial):
        s = self.slice_pool(sagittal)
        c = self.slice_pool(coronal)
        a = self.slice_pool(axial)
        fused = torch.cat([s, c, a], dim=1)
        return self.head(fused)          # (B, n_targets) raw logits

In [10]:
# Run once (Section B): precomputes + caches backbone features for every labeled study.
backbone = SliceBackbone(freeze=FREEZE_BACKBONE).to(device)
precompute_features(consensus_df['StudyInstanceUID'].tolist(), train_priority_series, backbone, FEATURE_CACHE_DIR)

4407 / 4407 studies need feature extraction (11 loader threads, batch size 16)
  16/4407 studies | 2s elapsed | 8.32 studies/s
  336/4407 studies | 25s elapsed | 13.53 studies/s
  656/4407 studies | 48s elapsed | 13.78 studies/s
  976/4407 studies | 71s elapsed | 13.78 studies/s
  1296/4407 studies | 95s elapsed | 13.67 studies/s
  1616/4407 studies | 117s elapsed | 13.77 studies/s
  1936/4407 studies | 139s elapsed | 13.89 studies/s
  2256/4407 studies | 162s elapsed | 13.96 studies/s
  2576/4407 studies | 183s elapsed | 14.04 studies/s
  2896/4407 studies | 205s elapsed | 14.11 studies/s
  3216/4407 studies | 229s elapsed | 14.06 studies/s
  3536/4407 studies | 251s elapsed | 14.07 studies/s
  3856/4407 studies | 274s elapsed | 14.09 studies/s
  4176/4407 studies | 296s elapsed | 14.12 studies/s
  4407/4407 studies | 312s elapsed | 14.12 studies/s
Precompute finished: 4407 studies in 312s -> D:/Coding/kaggle/RSNA/output/v2/feature_cache


## Section C: loss & metrics

In [11]:
def weighted_bce(logits, targets, row_weight, pos_weight):
    per_elem = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight, reduction='none')
    per_sample = per_elem.mean(dim=1)
    return (per_sample * row_weight).sum() / row_weight.sum()


def macro_auc(y_true_bin, y_prob, cols):
    """y_true_bin: (N, len(cols)) hard 0/1 labels. Skips columns with only one class present."""
    per_class = {}
    for i, c in enumerate(cols):
        yt = y_true_bin[:, i]
        if len(np.unique(yt)) < 2:
            continue
        per_class[c] = roc_auc_score(yt, y_prob[:, i])
    macro = float(np.mean(list(per_class.values()))) if per_class else float('nan')
    return macro, per_class

## Section D: train / eval loop functions

In [12]:
def train_one_epoch(model, loader, optimizer, pos_weight, scaler):
    model.train()
    total_loss = 0.0
    for batch in loader:
        sag, cor, ax = batch['sagittal'].to(device), batch['coronal'].to(device), batch['axial'].to(device)
        y, w = batch['labels'].to(device), batch['weight'].to(device)

        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, enabled=USE_AMP):
            logits = model(sag, cor, ax)
            loss = weighted_bce(logits, y, w, pos_weight)

        if USE_AMP:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * sag.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_probs, all_uids = [], []
    for batch in loader:
        sag, cor, ax = batch['sagittal'].to(device), batch['coronal'].to(device), batch['axial'].to(device)
        logits = model(sag, cor, ax)
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_uids.extend(batch['study_uid'])
    return np.concatenate(all_probs, axis=0), all_uids

## Section E: run training (per fold)

In [13]:
consensus_indexed = consensus_df.set_index('StudyInstanceUID')
gold_indexed = gold_df.set_index('StudyInstanceUID')

oof_probs = {}
fold_ckpts = {}

for fold in FOLDS_TO_RUN:
    print(f"\n{'=' * 72}\nFold {fold}\n{'=' * 72}")
    train_ids = consensus_df.loc[consensus_df.fold != fold, 'StudyInstanceUID'].tolist()
    val_ids = consensus_df.loc[consensus_df.fold == fold, 'StudyInstanceUID'].tolist()
    print(f'train studies: {len(train_ids)} | val studies: {len(val_ids)}')

    train_ds = KneeFeatureDataset(train_ids, FEATURE_CACHE_DIR,
                                   labels_df=consensus_df, gold_ids=gold_ids, gold_weight=GOLD_WEIGHT)
    val_ds = KneeFeatureDataset(val_ids, FEATURE_CACHE_DIR,
                                 labels_df=consensus_df, gold_ids=gold_ids, gold_weight=GOLD_WEIGHT)

    train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True,
                               num_workers=TRAIN_NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=False,
                             num_workers=TRAIN_NUM_WORKERS, pin_memory=True)

    prevalence = consensus_df.loc[consensus_df.fold != fold, TARGET_COLS].ge(0.5).mean().values
    pos_weight = torch.tensor(np.clip((1 - prevalence) / np.clip(prevalence, 1e-3, None), 1.0, 10.0),
                               dtype=torch.float32).to(device)

    model = KneeFeatureModel(feat_dim=1280, n_targets=len(TARGET_COLS)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_gold_auc = -1.0
    ckpt_path = os.path.join(OUTPUT_DIR, f'v2_fold{fold}.pt')

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, pos_weight, scaler)
        val_probs, val_uids = predict(model, val_loader)

        val_soft_true = consensus_indexed.loc[val_uids, TARGET_COLS].values
        noisy_auc, _ = macro_auc((val_soft_true >= 0.5).astype(int), val_probs, TARGET_COLS)

        gold_mask = np.array([u in gold_ids for u in val_uids])
        if gold_mask.any():
            gold_uids_here = [u for u, m in zip(val_uids, gold_mask) if m]
            gold_true = gold_indexed.loc[gold_uids_here, TARGET_COLS].values.astype(int)
            gold_auc, _ = macro_auc(gold_true, val_probs[gold_mask], TARGET_COLS)
        else:
            gold_auc = float('nan')

        marker = ''
        if not np.isnan(gold_auc) and gold_auc > best_gold_auc:
            best_gold_auc = gold_auc
            marker = ' *'
            torch.save(model.state_dict(), ckpt_path)

        print(f'epoch {epoch:>2}/{EPOCHS} | loss {train_loss:.4f} | '
              f'noisy AUC {noisy_auc:.4f} | gold AUC {gold_auc:.4f} | '
              f'{time.time() - t0:>5.1f}s{marker}')

    if not os.path.exists(ckpt_path):     # fallback: fold had no gold rows to select on
        torch.save(model.state_dict(), ckpt_path)

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    val_probs, val_uids = predict(model, val_loader)
    for u, p in zip(val_uids, val_probs):
        oof_probs[u] = p
    fold_ckpts[fold] = ckpt_path
    print(f'Fold {fold} best gold AUC: {best_gold_auc:.4f} -> {ckpt_path}')


Fold 0
train studies: 3525 | val studies: 882


C:\Users\nishk\AppData\Local\Temp\ipykernel_32796\1463495455.py:29: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


epoch  1/20 | loss 1.0096 | noisy AUC 0.7367 | gold AUC 0.6465 |   7.1s *
epoch  2/20 | loss 0.9559 | noisy AUC 0.7576 | gold AUC 0.7242 |   4.8s *
epoch  3/20 | loss 0.9299 | noisy AUC 0.7657 | gold AUC 0.7227 |   6.1s
epoch  4/20 | loss 0.9178 | noisy AUC 0.7721 | gold AUC 0.7601 |   4.7s *
epoch  5/20 | loss 0.8989 | noisy AUC 0.7740 | gold AUC 0.7307 |   4.5s
epoch  6/20 | loss 0.8846 | noisy AUC 0.7746 | gold AUC 0.7475 |   5.2s
epoch  7/20 | loss 0.8701 | noisy AUC 0.7750 | gold AUC 0.7226 |   5.7s
epoch  8/20 | loss 0.8614 | noisy AUC 0.7799 | gold AUC 0.7394 |   5.9s
epoch  9/20 | loss 0.8472 | noisy AUC 0.7757 | gold AUC 0.7425 |   6.0s
epoch 10/20 | loss 0.8353 | noisy AUC 0.7709 | gold AUC 0.7382 |   4.8s
epoch 11/20 | loss 0.8268 | noisy AUC 0.7808 | gold AUC 0.7419 |   4.8s
epoch 12/20 | loss 0.8142 | noisy AUC 0.7752 | gold AUC 0.7581 |   4.5s
epoch 13/20 | loss 0.7992 | noisy AUC 0.7728 | gold AUC 0.7471 |   4.6s
epoch 14/20 | loss 0.7891 | noisy AUC 0.7745 | gold AUC 0.

## Section F: out-of-fold summary

In [14]:
oof_uids = list(oof_probs.keys())
oof_matrix = np.stack([oof_probs[u] for u in oof_uids])

oof_soft_true = consensus_indexed.loc[oof_uids, TARGET_COLS].values
noisy_macro, noisy_per_class = macro_auc((oof_soft_true >= 0.5).astype(int), oof_matrix, TARGET_COLS)

gold_mask = np.array([u in gold_ids for u in oof_uids])
gold_oof_uids = [u for u, m in zip(oof_uids, gold_mask) if m]
gold_true = gold_indexed.loc[gold_oof_uids, TARGET_COLS].values.astype(int)
gold_macro, gold_per_class = macro_auc(gold_true, oof_matrix[gold_mask], TARGET_COLS)

print(f'OOF studies: {len(oof_uids)} (of which genuinely-labeled: {len(gold_oof_uids)})')
print(f'Noisy macro AUC  (vs. consensus soft pseudo-labels): {noisy_macro:.4f}')
print(f'HONEST macro AUC (vs. the 58 real gold labels):      {gold_macro:.4f}   <-- compare against v1\'s 0.7538')

print('\nPer-class honest AUC:')
for c, v in sorted(gold_per_class.items(), key=lambda x: -x[1]):
    print(f'  {c:<20} {v:.4f}')

OOF studies: 882 (of which genuinely-labeled: 13)
Noisy macro AUC  (vs. consensus soft pseudo-labels): 0.7721
HONEST macro AUC (vs. the 58 real gold labels):      0.7601   <-- compare against v1's 0.7538

Per-class honest AUC:
  Lateral OA           1.0000
  Baker's              0.9444
  Contusion            0.9286
  MCL                  0.9091
  Effusion             0.8056
  PF OA                0.8000
  Medial OA            0.7778
  Fracture             0.6944
  ACL                  0.6667
  Medial Meniscus      0.6429
  Lateral Meniscus     0.5476
  Synovitis            0.4048


## Section G: inference on test set + submission file

Test studies were already preprocessed in Section A (from `TEST_DICOM_DIR`, `skip_existing=False`
so a stale run never lingers), so this is now just feature extraction + prediction -- no
DICOM handling left in the inference path at all.

In [15]:
test_df = pd.read_csv(TEST_CSV)

precompute_features(test_df['StudyInstanceUID'].tolist(), test_priority_series, backbone, FEATURE_CACHE_DIR,
                    skip_existing=False)

test_ds = KneeFeatureDataset(test_df['StudyInstanceUID'].tolist(), FEATURE_CACHE_DIR)
test_loader = DataLoader(test_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=False, num_workers=TRAIN_NUM_WORKERS)

fold_preds = []
test_uids = None
for fold, ckpt_path in fold_ckpts.items():
    model = KneeFeatureModel(feat_dim=1280, n_targets=len(TARGET_COLS)).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    probs, uids = predict(model, test_loader)
    fold_preds.append(probs)
    test_uids = uids

final_probs = np.mean(fold_preds, axis=0)
sub = pd.DataFrame(final_probs, columns=TARGET_COLS)
sub.insert(0, 'StudyInstanceUID', test_uids)
sub = sub[pd.read_csv(SAMPLE_SUB_CSV).columns]   # match exact column order

sub_path = os.path.join(OUTPUT_DIR, 'submission.csv')   # Kaggle requires this exact filename
sub.to_csv(sub_path, index=False)
print(f'Saved: {sub_path}')
sub.head()

3 / 3 studies need feature extraction (11 loader threads, batch size 16)
  3/3 studies | 1s elapsed | 5.06 studies/s
Precompute finished: 3 studies in 1s -> D:/Coding/kaggle/RSNA/output/v2/feature_cache
Saved: D:/Coding/kaggle/RSNA/output/v2\submission.csv


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.380520,0.462563,0.392633,0.419271,0.525935,0.480213,0.481776,0.456770,0.592745,0.424282,0.415413,0.359588
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.660015,0.593605,0.486585,0.597782,0.621546,0.620860,0.589127,0.696388,0.846652,0.492662,0.584887,0.489844
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.231910,0.412958,0.359869,0.377737,0.453039,0.402463,0.496750,0.318148,0.400138,0.404273,0.247820,0.373114


## What v3 should add

- Pool over **all** series in a plane, not just one priority series per plane -- the biggest
  known modeling gap carried over from v1.
- Per-label learned attention over the 3 plane embeddings instead of flat concatenation (the
  "region-guided" idea in `docs/model_architecture_plan.md`).
- Slice-level augmentation -- currently dropped, same tradeoff as v1 (precomputed features
  are deterministic). Reintroduce once the backbone is unfrozen for Phase-2 fine-tuning, or
  precompute a flipped variant and sample between the two at train time.
- If disk space on Kaggle becomes tight, `PREPROCESSED_DIR` can be deleted after Section B's
  feature cache is built -- only the (much smaller) feature cache is needed for training/CV.
- Consider Nyúl-style histogram standardization instead of per-series percentile clipping if
  cross-scanner intensity variation turns out to matter (the `manufacturer`/
  `magnetic_field_strength` columns in `preprocessing_metadata.csv` are there to check this).
- Widen `FOLDS_TO_RUN` to all 5 once this is validated.